# PIPELINE CREATION, RETRAINING AND EXECUTION SCRIPTS

## PRODUCTION PIPELINE CREATION & INITIAL EXPORT

In [6]:
import os
import pandas as pd
import cloudpickle
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

# --- Configuration ---
PROJECT_PATH = '/Users/rober/smartport-ai-risk-early-warning'
MODELS_PATH = os.path.join(PROJECT_PATH, '04_Models')
os.makedirs(MODELS_PATH, exist_ok=True)

# 1. Load Balanced Data (CSV-based to avoid NumPy 2.0 conflicts)
X_path = os.path.join(PROJECT_PATH, '02_Data/03_Working/X_balanced.csv')
y_path = os.path.join(PROJECT_PATH, '02_Data/03_Working/y_balanced.csv')

X = pd.read_csv(X_path).astype('float32')
y = pd.read_csv(y_path).squeeze()

# 2. Build Fresh Pipeline
# We use SimpleImputer for production robustness and XGBoost for classification
pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('model', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

# 3. Initial Training
print("Training production pipeline...")
pipe.fit(X, y)

# 4. Serialization (Using cloudpickle for Python 3.13 compatibility)
EXECUTION_PIPE = os.path.join(MODELS_PATH, 'pipe_execution.pkl')
FEATURES_FILE = os.path.join(MODELS_PATH, 'model_features.pkl')

with open(EXECUTION_PIPE, 'wb') as f:
    cloudpickle.dump(pipe, f)
    
with open(FEATURES_FILE, 'wb') as f:
    cloudpickle.dump(X.columns.tolist(), f)

print(f"✅ Success: Pipeline and Features exported to {MODELS_PATH}")

Training production pipeline...


/Users/rober/ai-corporate-suite/venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [11:52:30] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Success: Pipeline and Features exported to /Users/rober/smartport-ai-risk-early-warning/04_Models


## Automated Model Retraining & Quality Audit

In [7]:
import cloudpickle
from sklearn.metrics import recall_score

def run_retraining():
    print("Starting automated retraining process...")
    
    # 1. Load existing execution pipeline
    model_path = os.path.join(MODELS_PATH, 'pipe_execution.pkl')
    with open(model_path, 'rb') as f:
        pipe = cloudpickle.load(f)
    
    # 2. Re-fit pipeline on current balanced data
    print("Refitting model on new data...")
    pipe.fit(X, y)
    
    # 3. Quality Audit: Ensure Recall (Sensitivity) is above 90%
    # In maritime risk, catching delays is more critical than false alarms
    y_pred = pipe.predict(X)
    recall = recall_score(y, y_pred)
    
    if recall >= 0.90:
        with open(model_path, 'wb') as f:
            cloudpickle.dump(pipe, f)
        print(f"🚀 Retraining Successful! Model updated with Recall: {recall:.4f}")
    else:
        print(f"⚠️ Update Rejected: Recall {recall:.4f} is below the 0.90 threshold.")

if __name__ == "__main__":
    run_retraining()

Starting automated retraining process...
Refitting model on new data...


/Users/rober/ai-corporate-suite/venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [11:53:02] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🚀 Retraining Successful! Model updated with Recall: 0.9646


## PRODUCTION EXECUTION SCRIPT (SCALED FOR BUSINESS)

In [18]:
import os
import pandas as pd
import cloudpickle
from datetime import datetime

def run_execution():
    print("🚀 Starting Production Execution...")
    
    # --- Configuration ---
    PROJECT_PATH = '/Users/rober/smartport-ai-risk-early-warning'
    MODEL_FILE = os.path.join(PROJECT_PATH, '04_Models/pipe_execution.pkl')
    FEATURES_FILE = os.path.join(PROJECT_PATH, '04_Models/model_features.pkl')
    SOURCE_DATA = os.path.join(PROJECT_PATH, '02_Data/03_Working/work_fs.csv')
    OUTPUT_CSV = os.path.join(PROJECT_PATH, '05_Outputs/risk_alerts.csv')

    # --- Load Model and Features ---
    with open(MODEL_FILE, 'rb') as f:
        pipe = cloudpickle.load(f)
    with open(FEATURES_FILE, 'rb') as f:
        expected_features = cloudpickle.load(f)
    
    # --- Data Loading and Alignment ---
    df_live = pd.read_csv(SOURCE_DATA)
    X_live = df_live.reindex(columns=expected_features, fill_value=0).astype('float32')
    
    # 1. Get raw probabilities from the XGBoost model
    raw_scores = pipe.predict_proba(X_live)[:, 1]
    
    # 2. Build Results DataFrame and Sort by Risk (Ranking)
    results = pd.DataFrame({'vessel_id': df_live.index, 'raw_score': raw_scores})
    results = results.sort_values(by='raw_score', ascending=False)
    
    # 3. FORCED CATEGORIZATION STRATEGY (Operational Calibration)
    # Initialize all records as NORMAL
    results['risk_score'] = 0.15
    results['risk_level'] = 'NORMAL'
    
    # Assign WARNING status (Rank positions 101 to 1000)
    results.iloc[100:1000, results.columns.get_loc('risk_score')] = 0.65
    results.iloc[100:1000, results.columns.get_loc('risk_level')] = 'WARNING'
    
    # Assign CRITICAL status (Top 100 offenders)
    results.iloc[0:100, results.columns.get_loc('risk_score')] = 0.92
    results.iloc[0:100, results.columns.get_loc('risk_level')] = 'CRITICAL'
    
    # --- Action Mapping and Metadata ---
    ACTION_MAP = {
        'CRITICAL': 'Immediate intervention (reassign berth)',
        'WARNING': 'Monitor ETA and AIS stability closely',
        'NORMAL': 'Routine operations'
    }
    results['recommended_action'] = results['risk_level'].map(ACTION_MAP)
    results['timestamp'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # --- Export Results ---
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    results.to_csv(OUTPUT_CSV, index=False)
    
    print(f"✅ Execution Complete.")
    print(f"📊 Distribution check: {results['risk_level'].value_counts().to_dict()}")

if __name__ == "__main__":
    run_execution()

🚀 Starting Production Execution...
✅ Execution Complete.
📊 Distribution check: {'NORMAL': 115481, 'WARNING': 900, 'CRITICAL': 100}
